In [ ]:
from huggingface_hub import hf_hub_download
import pandas as pd

path = hf_hub_download(
    repo_id="jumelet/multiblimp",
    filename="uzb/data.tsv",
    repo_type="dataset"
)

df = pd.read_csv(path, sep="\t")

print("Total rows:", len(df))
print("Columns:", df.columns.tolist())

display(df[["sen", "wrong_sen"]].head(10))

In [ ]:
from datasets import load_dataset

dataset = load_dataset("tahrirchi/uzlib")

print(dataset)

for split in dataset:
    print(split, len(dataset[split]))

In [ ]:
import torch
import pandas as pd
import json
import numpy as np

from pathlib import Path
from tqdm.auto import tqdm
from datetime import datetime
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from huggingface_hub import hf_hub_download

assert torch.cuda.is_available(), "CUDA topilmadi!"

DEVICE = "cuda"

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
print("BF16:", torch.cuda.is_bf16_supported())

vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"VRAM: {vram:.2f} GiB")

In [ ]:
from pathlib import Path
import pandas as pd

dataset_files = list(Path(".").rglob("cleaned_002.jsonl"))

assert dataset_files, "Dataset topilmadi!"

DATASET_PATH = dataset_files[0]

raw_df = pd.read_json(DATASET_PATH, lines=True)

print("Dataset path:", DATASET_PATH)
print("Total rows:", len(raw_df))
print("Columns:", raw_df.columns.tolist())

display(raw_df.head(3))

In [ ]:
import json
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm

dataset_files = list(Path(".").rglob("cleaned_002.jsonl"))

assert dataset_files, "Dataset topilmadi!"

DATASET_PATH = dataset_files[0]

valid_rows = []
invalid_rows = []
total_rows = 0

with open(DATASET_PATH, "r", encoding="utf-8") as f:

    for line_no, line in enumerate(
        tqdm(f, desc="Reading dataset"),
        start=1
    ):
        total_rows += 1

        try:
            row = json.loads(line)

            if not isinstance(row, dict):
                raise ValueError("Row is not a JSON object")

            valid_rows.append(row)

        except (ValueError, TypeError) as e:
            invalid_rows.append({
                "line": line_no,
                "error": str(e),
                "preview": line[:300]
            })

raw_df = pd.DataFrame(valid_rows)

print("=" * 60)
print("DATASET REPORT")
print("=" * 60)

print("Dataset:", DATASET_PATH)
print(f"Total rows:   {total_rows:,}")
print(f"Valid rows:   {len(valid_rows):,}")
print(f"Invalid rows: {len(invalid_rows):,}")
print("Columns:", raw_df.columns.tolist())

print("=" * 60)

display(raw_df.head(3))

In [ ]:
if invalid_rows:

    errors_df = pd.DataFrame(invalid_rows)

    display(errors_df.head(10))

    errors_df.to_json(
        "invalid_rows.jsonl",
        orient="records",
        lines=True,
        force_ascii=False
    )

    print("Invalid rows saved.")

else:
    print("All rows are valid!")

In [ ]:
print("Available columns:", raw_df.columns.tolist())

TEXT_COLUMN = "text"

assert TEXT_COLUMN in raw_df.columns

print("Empty texts:", raw_df[TEXT_COLUMN].isna().sum())

raw_df = raw_df[
    raw_df[TEXT_COLUMN].apply(
        lambda x: isinstance(x, str) and bool(x.strip())
    )
].copy()

before = len(raw_df)

raw_df = raw_df.drop_duplicates(
    subset=[TEXT_COLUMN]
).reset_index(drop=True)

print("Duplicates removed:", before - len(raw_df))
print("Valid unique rows:", len(raw_df))

lengths = raw_df[TEXT_COLUMN].str.len()

print(f"Average characters: {lengths.mean():.2f}")
print(f"Min characters: {lengths.min()}")
print(f"Max characters: {lengths.max()}")
print(f"Total characters: {lengths.sum():,}")

In [ ]:
import json
import os
from pathlib import Path

DATASET_PATH = Path("Cleaned 50k Raw/cleaned_002.jsonl")

assert DATASET_PATH.is_file(), "Dataset topilmadi!"

TEMP_PATH = DATASET_PATH.with_suffix(".jsonl.tmp")

valid_count = 0
invalid_count = 0
removed_lines = []

with open(DATASET_PATH, "r", encoding="utf-8") as src, \
     open(TEMP_PATH, "w", encoding="utf-8") as dst:

    for line_no, line in enumerate(src, start=1):

        try:
            row = json.loads(line)

            if not isinstance(row, dict):
                raise ValueError("Invalid JSON object")

            dst.write(line.rstrip("\r\n") + "\n")
            valid_count += 1

        except (ValueError, TypeError):
            invalid_count += 1
            removed_lines.append(line_no)

# Original faylni almashtirishdan oldin tekshirish
assert valid_count == 50197
assert invalid_count == 4

# Original datasetni tozalangan fayl bilan almashtirish
os.replace(TEMP_PATH, DATASET_PATH)

print("=" * 60)
print("DATASET CLEANING COMPLETED")
print("=" * 60)

print(f"Valid rows:    {valid_count:,}")
print(f"Removed rows:  {invalid_count:,}")
print(f"Removed lines: {removed_lines}")
print(f"Saved to:      {DATASET_PATH}")

In [ ]:
valid_count = 0
invalid_count = 0

with open(DATASET_PATH, "r", encoding="utf-8") as f:

    for line in f:
        try:
            assert isinstance(json.loads(line), dict)
            valid_count += 1
        except (ValueError, AssertionError):
            invalid_count += 1

print(f"Total rows:   {valid_count + invalid_count:,}")
print(f"Valid rows:   {valid_count:,}")
print(f"Invalid rows: {invalid_count:,}")

assert invalid_count == 0
assert valid_count == 50197

print("INTEGRITY CHECK: PASS")

In [ ]:
from sklearn.model_selection import train_test_split
from pathlib import Path
import json

SEED = 42

train_df, temp_df = train_test_split(
    raw_df,
    test_size=0.10,
    random_state=SEED
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED
)

print(f"Train:      {len(train_df):,}")
print(f"Validation: {len(valid_df):,}")
print(f"Test:       {len(test_df):,}")

In [ ]:
SPLIT_DIR = Path("./data_splits")
SPLIT_DIR.mkdir(exist_ok=True)

for name, df in [
    ("train", train_df),
    ("validation", valid_df),
    ("test", test_df)
]:
    path = SPLIT_DIR / f"{name}.jsonl"

    df.to_json(
        path,
        orient="records",
        lines=True,
        force_ascii=False
    )

    print(f"Saved: {path} | Rows: {len(df):,}")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Tokenizer:", MODEL_ID)
print("Vocabulary size:", len(tokenizer))
print("EOS token:", tokenizer.eos_token)
print("PAD token:", tokenizer.pad_token)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
import numpy as np
from tqdm.auto import tqdm

def calculate_token_lengths(texts, batch_size=64):

    lengths = []

    for start in tqdm(
        range(0, len(texts), batch_size),
        desc="Counting tokens"
    ):

        batch = texts[start:start + batch_size]

        encoded = tokenizer(
            batch,
            add_special_tokens=False,
            truncation=False,
            return_attention_mask=False,
            return_length=True,
            verbose=False
        )

        lengths.extend(encoded["length"])

    return np.array(lengths, dtype=np.int64)


train_lengths = calculate_token_lengths(
    train_df["text"].tolist()
)

print("=" * 60)
print("TRAIN DATASET TOKEN STATISTICS")
print("=" * 60)

print(f"Documents:    {len(train_lengths):,}")
print(f"Total tokens: {train_lengths.sum():,}")
print(f"Mean tokens:  {train_lengths.mean():,.2f}")
print(f"Min tokens:   {train_lengths.min():,}")
print(f"Max tokens:   {train_lengths.max():,}")

for p in [25, 50, 75, 90, 95, 99]:
    print(f"P{p}: {np.percentile(train_lengths, p):,.0f}")

In [ ]:
import pandas as pd

statistics = []

for seq_len in [128, 256, 512, 1024, 2048]:

    statistics.append({
        "sequence_length": seq_len,

        "documents_shorter_or_equal": int(
            (train_lengths <= seq_len).sum()
        ),

        "documents_longer": int(
            (train_lengths > seq_len).sum()
        ),

        "coverage_percent": round(
            (train_lengths <= seq_len).mean() * 100,
            2
        ),

        "estimated_chunks": int(
            np.ceil(train_lengths / seq_len).sum()
        )
    })

display(pd.DataFrame(statistics))

In [ ]:
import torch

assert torch.cuda.is_available(), "CUDA not available!"

DEVICE = "cuda"

DTYPE = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE
).to(DEVICE)

model.eval()

total_params = sum(p.numel() for p in model.parameters())

print("=" * 60)
print("BASE MODEL INFORMATION")
print("=" * 60)

print("Model:", MODEL_ID)
print("GPU:", torch.cuda.get_device_name(0))
print("Precision:", DTYPE)
print(f"Parameters: {total_params:,}")
print("Hidden size:", model.config.hidden_size)
print("Layers:", model.config.num_hidden_layers)
print("Attention heads:", model.config.num_attention_heads)

print(
    f"VRAM allocated: "
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GiB"
)

In [ ]:
%pip install unsloth
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-0.5B",
    max_seq_length=256,
    dtype=torch.bfloat16,
    load_in_4bit=False,
    load_in_8bit=False,
    full_finetuning=True,
)

model.config.use_cache = False
model.gradient_checkpointing_enable()

In [ ]:
total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {total_params - trainable_params:,}")

print(
    f"Trainable percent: "
    f"{100 * trainable_params / total_params:.2f}%"
)

assert trainable_params == total_params

In [ ]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import unsloth
from unsloth import FastLanguageModel

import torch
import transformers
import bitsandbytes

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-0.5B"
MAX_SEQ_LENGTH = 256

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.bfloat16,

    load_in_4bit=False,
    load_in_8bit=False,
    full_finetuning=True,
)

model.config.use_cache = False

model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    }
)

print("MODEL LOADED SUCCESSFULLY")

In [ ]:
total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

free_vram, total_vram = torch.cuda.mem_get_info()

print("=" * 60)
print("MODEL REPORT")
print("=" * 60)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {total_params - trainable_params:,}")
print(f"Trainable percentage: {trainable_params / total_params * 100:.2f}%")

print("Model dtype:", next(model.parameters()).dtype)
print("Device:", next(model.parameters()).device)

print(f"Allocated VRAM: {torch.cuda.memory_allocated() / 1024**3:.3f} GiB")
print(f"Reserved VRAM:  {torch.cuda.memory_reserved() / 1024**3:.3f} GiB")
print(f"Free VRAM:      {free_vram / 1024**3:.3f} GiB")
print(f"Total VRAM:     {total_vram / 1024**3:.3f} GiB")

assert trainable_params == total_params

print("FULL FINE-TUNING CHECK: PASS")

In [ ]:
import json
import pandas as pd
import numpy as np

from pathlib import Path
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

# MultiBLiMP Uzbek
blimp_path = hf_hub_download(
    repo_id="jumelet/multiblimp",
    filename="uzb/data.tsv",
    repo_type="dataset"
)

blimp_df = pd.read_csv(blimp_path, sep="\t")

# UzLiB
uzlib_dataset = load_dataset("tahrirchi/uzlib")

uzlib_df = pd.DataFrame([
    {**row, "category": split}
    for split in uzlib_dataset
    for row in uzlib_dataset[split]
])

# Test dataset
test_df = pd.read_json(
    "data_splits/test.jsonl",
    lines=True
)

print("MultiBLiMP:", len(blimp_df))
print("UzLiB:", len(uzlib_df))
print("Test documents:", len(test_df))

print("\nMultiBLiMP columns:", blimp_df.columns.tolist())
print("UzLiB columns:", uzlib_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())

display(blimp_df.head(3))
display(uzlib_df.head(3))

In [ ]:
FastLanguageModel.for_inference(model)

model.eval()

print("Training mode:", model.training)
print("Model dtype:", next(model.parameters()).dtype)
print("Device:", next(model.parameters()).device)

In [ ]:
import torch
import torch.nn.functional as F

@torch.inference_mode()
def score_sentence(text):

    input_ids = tokenizer(
        text,
        return_tensors="pt",
        add_special_tokens=False
    )["input_ids"].to("cuda")

    assert input_ids.shape[1] >= 2

    outputs = model(
        input_ids=input_ids,
        use_cache=False
    )

    logits = outputs.logits[:, :-1, :].float()
    labels = input_ids[:, 1:]

    nll = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        labels.reshape(-1),
        reduction="none"
    )

    return {
        "logprob": -nll.sum().item(),
        "mean_logprob": -nll.mean().item(),
        "tokens": labels.numel()
    }

In [ ]:
sentences = [
    ("Men bugun universitetga bordim.",
     "Men bugun universitetga bordi."),

    ("Men kitobni o'qidim.",
     "Men kitobga o'qidim."),

    ("Biz ertaga Toshkentga boramiz.",
     "Biz ertaga Toshkentga boraman."),

    ("Mening kitobim stol ustida turibdi.",
     "Mening kitobing stol ustida turibdi.")
]

for correct, incorrect in sentences:

    good = score_sentence(correct)
    bad = score_sentence(incorrect)

    passed = good["logprob"] > bad["logprob"]

    print("=" * 60)
    print("Correct:", correct)
    print("Score:", good["logprob"])

    print("Incorrect:", incorrect)
    print("Score:", bad["logprob"])

    print("Result:", "PASS" if passed else "FAIL")

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

blimp_results = []

for idx, row in tqdm(
    blimp_df.iterrows(),
    total=len(blimp_df),
    desc="MultiBLiMP Uzbek"
):

    correct = row["sen"]
    incorrect = row["wrong_sen"]

    good = score_sentence(correct)
    bad = score_sentence(incorrect)

    blimp_results.append({
        "id": idx,

        "correct_sentence": correct,
        "incorrect_sentence": incorrect,

        "correct_logprob": good["logprob"],
        "incorrect_logprob": bad["logprob"],

        "correct_mean_logprob": good["mean_logprob"],
        "incorrect_mean_logprob": bad["mean_logprob"],

        "correct_tokens": good["tokens"],
        "incorrect_tokens": bad["tokens"],

        "margin": good["logprob"] - bad["logprob"],

        "passed": good["logprob"] > bad["logprob"],

        "normalized_passed": (
            good["mean_logprob"] > bad["mean_logprob"]
        )
    })

blimp_results_df = pd.DataFrame(blimp_results)

print("=" * 60)
print("BASE MODEL — MULTIBLIMP UZBEK")
print("=" * 60)

print("Model:", MODEL_ID)
print("Total pairs:", len(blimp_results_df))
print("PASS:", blimp_results_df["passed"].sum())
print("FAIL:", (~blimp_results_df["passed"]).sum())

print(
    f"Accuracy: "
    f"{blimp_results_df['passed'].mean() * 100:.2f}%"
)

print(
    f"Normalized Accuracy: "
    f"{blimp_results_df['normalized_passed'].mean() * 100:.2f}%"
)

print(
    f"Average Margin: "
    f"{blimp_results_df['margin'].mean():.4f}"
)

In [ ]:
import json
from pathlib import Path

RESULTS_DIR = Path("./benchmark_results")
RESULTS_DIR.mkdir(exist_ok=True)

baseline_blimp = {
    "model": MODEL_ID,
    "stage": "base",
    "benchmark": "MultiBLiMP Uzbek",
    "total_pairs": len(blimp_results_df),
    "passed": int(blimp_results_df["passed"].sum()),
    "failed": int((~blimp_results_df["passed"]).sum()),
    "accuracy": float(
        blimp_results_df["passed"].mean() * 100
    ),
    "normalized_accuracy": float(
        blimp_results_df["normalized_passed"].mean() * 100
    ),
    "average_margin": float(
        blimp_results_df["margin"].mean()
    )
}

with open(
    RESULTS_DIR / "base_multiblimp_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(baseline_blimp, f, indent=4)

blimp_results_df.to_json(
    RESULTS_DIR / "base_multiblimp_details.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

print(json.dumps(baseline_blimp, indent=4))
print("\nBASELINE SAVED SUCCESSFULLY")

In [ ]:
import torch

LABELS = ["A", "B", "C", "D"]

LABEL_IDS = []

for label in LABELS:
    ids = tokenizer.encode(
        " " + label,
        add_special_tokens=False
    )

    assert len(ids) == 1, (label, ids)
    LABEL_IDS.append(ids[0])


@torch.inference_mode()
def evaluate_uzlib_question(row):

    prompt = (
        f"Savol: {row['question']}\n"
        f"A) {row['option_a']}\n"
        f"B) {row['option_b']}\n"
        f"C) {row['option_c']}\n"
        f"D) {row['option_d']}\n"
        "Javob:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to("cuda")

    outputs = model(
        **inputs,
        use_cache=False
    )

    logits = outputs.logits[0, -1].float()

    scores = logits[LABEL_IDS]

    probabilities = torch.softmax(scores, dim=-1)

    predicted = LABELS[probabilities.argmax().item()]
    correct = str(row["answer"]).strip().upper()

    assert correct in LABELS

    return {
        "predicted": predicted,
        "correct": correct,
        "passed": predicted == correct,

        **{
            label: probabilities[i].item()
            for i, label in enumerate(LABELS)
        }
    }

In [ ]:
for _, row in uzlib_df.head(5).iterrows():

    result = evaluate_uzlib_question(row)

    print("=" * 60)
    print("Question:", row["question"])

    for label in LABELS:
        print(
            f"{label}) {row[f'option_{label.lower()}']}"
            f" | Probability: {result[label]:.4f}"
        )

    print("Expected:", result["correct"])
    print("Predicted:", result["predicted"])
    print("Result:", "PASS" if result["passed"] else "FAIL")

In [ ]:
uzlib_results = []

for idx, row in tqdm(
    uzlib_df.iterrows(),
    total=len(uzlib_df),
    desc="UzLiB Benchmark"
):

    result = evaluate_uzlib_question(row)

    uzlib_results.append({
        "id": row["id"],
        "category": row["type"],
        "question": row["question"],
        **result
    })

uzlib_results_df = pd.DataFrame(uzlib_results)

uzlib_summary = (
    uzlib_results_df
    .groupby("category")
    .agg(
        total=("passed", "size"),
        correct=("passed", "sum"),
        accuracy=("passed", "mean")
    )
)

uzlib_summary["accuracy"] *= 100

overall_accuracy = (
    uzlib_results_df["passed"].mean() * 100
)

print("=" * 60)
print("BASE MODEL — UZLIB BENCHMARK")
print("=" * 60)

print("Model:", MODEL_ID)
print("Total:", len(uzlib_results_df))
print("PASS:", uzlib_results_df["passed"].sum())
print("FAIL:", (~uzlib_results_df["passed"]).sum())
print(f"Overall accuracy: {overall_accuracy:.2f}%")

display(uzlib_summary)

In [ ]:
baseline_uzlib = {
    "model": MODEL_ID,
    "stage": "base",
    "benchmark": "UzLiB",
    "total_questions": len(uzlib_results_df),
    "passed": int(uzlib_results_df["passed"].sum()),
    "failed": int((~uzlib_results_df["passed"]).sum()),
    "accuracy": float(overall_accuracy),
    "categories": json.loads(
        uzlib_summary.to_json(orient="index")
    ),
    "scoring_method": "zero_shot_next_token_ABCD"
}

with open(
    RESULTS_DIR / "base_uzlib_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        baseline_uzlib,
        f,
        indent=4,
        ensure_ascii=False
    )

uzlib_results_df.to_json(
    RESULTS_DIR / "base_uzlib_details.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

print(json.dumps(
    baseline_uzlib,
    indent=4,
    ensure_ascii=False
))

print("\nBASELINE SAVED SUCCESSFULLY")

In [ ]:
import math
import torch
import torch.nn.functional as F

from tqdm.auto import tqdm
from pathlib import Path

MAX_LENGTH = 256

@torch.inference_mode()
def evaluate_perplexity(model, tokenizer, texts):

    model.eval()
    device = next(model.parameters()).device

    total_nll = 0.0
    total_tokens = 0
    processed_documents = 0

    torch.cuda.reset_peak_memory_stats()

    for text in tqdm(
        texts,
        desc="Uzbek Perplexity Benchmark"
    ):

        if not isinstance(text, str) or not text.strip():
            continue

        tokens = tokenizer.encode(
            text,
            add_special_tokens=False,
            verbose=False
        )

        if len(tokens) < 2:
            continue

        for start in range(
            0, len(tokens) - 1, MAX_LENGTH - 1
        ):

            chunk = tokens[start:start + MAX_LENGTH]

            if len(chunk) < 2:
                continue

            input_ids = torch.tensor(
                [chunk],
                dtype=torch.long,
                device=device
            )

            outputs = model(
                input_ids=input_ids,
                use_cache=False
            )

            logits = outputs.logits[:, :-1, :].float()
            labels = input_ids[:, 1:]

            nll = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
                reduction="sum"
            )

            total_nll += nll.item()
            total_tokens += labels.numel()

        processed_documents += 1

    assert total_tokens > 0

    avg_loss = total_nll / total_tokens

    return {
        "documents": processed_documents,
        "tokens": total_tokens,
        "loss": avg_loss,
        "perplexity": math.exp(avg_loss),
        "sequence_length": MAX_LENGTH,
        "peak_vram_gib": (
            torch.cuda.max_memory_allocated() / 1024**3
        )
    }

In [ ]:
base_ppl_results = evaluate_perplexity(
    model,
    tokenizer,
    test_df["text"].tolist()
)

print("=" * 60)
print("BASE MODEL — UZBEK PERPLEXITY")
print("=" * 60)

print(json.dumps(base_ppl_results, indent=4))


In [ ]:
import hashlib

test_path = Path("data_splits/test.jsonl")

baseline_ppl = {
    "model": MODEL_ID,
    "stage": "base",
    "benchmark": "Uzbek Perplexity",
    "test_sha256": hashlib.sha256(
        test_path.read_bytes()
    ).hexdigest(),
    **base_ppl_results
}

with open(
    RESULTS_DIR / "base_perplexity_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(baseline_ppl, f, indent=4)

print("BASELINE PERPLEXITY SAVED SUCCESSFULLY")

In [ ]:
PROMPTS = [
    # General knowledge
    "O‘zbekiston Respublikasi Markaziy Osiyoda",
    "Buxoro shahri qadim zamonlardan",
    "O‘zbek tilining grammatik tuzilishi",
    "O‘zbekistonning davlat tili",

    # Science and technology
    "Sun'iy intellekt texnologiyalari",
    "Dasturlash tillari yordamida",
    "Matematika fanining asosiy vazifasi",
    "Kompyuter tizimlarida ma'lumotlar",

    # Everyday language
    "Men bugun ertalab uyg‘onib",
    "Universitet talabalari har kuni",
    "Kecha do‘stlarim bilan",
    "Onam menga har doim",

    # Literature and nature
    "Bahor fasli kelishi bilan",
    "Quyosh tog‘lar ortidan",
    "Bir kuni kichkina qishloqda",
    "Kuz faslida daraxtlarning",

    # Education and society
    "Kitob o‘qish insonning",
    "Zamonaviy ta'lim tizimida",
    "O‘zbek xalqining milliy qadriyatlari",
    "Atrof-muhitni muhofaza qilish uchun"
]

print("Total prompts:", len(PROMPTS))

In [ ]:
import torch
import time

from tqdm.auto import tqdm

MAX_NEW_TOKENS = 100

@torch.inference_mode()
def evaluate_completion(model, tokenizer, prompts):

    model.eval()
    device = next(model.parameters()).device

    results = []

    for prompt in tqdm(prompts, desc="Text Completion"):

        inputs = tokenizer(
            prompt,
            return_tensors="pt"
        ).to(device)

        torch.cuda.synchronize()
        start_time = time.perf_counter()

        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.0,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id
        )

        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start_time

        new_tokens = outputs[
            0, inputs["input_ids"].shape[1]:
        ]

        completion = tokenizer.decode(
            new_tokens,
            skip_special_tokens=True
        )

        results.append({
            "prompt": prompt,
            "completion": completion,
            "generated_tokens": len(new_tokens),
            "elapsed_seconds": elapsed,
            "tokens_per_second": len(new_tokens) / elapsed
        })

    return results

In [ ]:
base_completion_results = evaluate_completion(
    model,
    tokenizer,
    PROMPTS
)

for idx, row in enumerate(
    base_completion_results,
    start=1
):
    print("=" * 80)

    print(f"EXAMPLE: {idx}")

    print("\nPROMPT:")
    print(row["prompt"])

    print("\nCOMPLETION:")
    print(row["completion"])

    print(
        f"\nSpeed: "
        f"{row['tokens_per_second']:.2f} tokens/sec"
    )

In [ ]:
baseline_completion = {
    "model": MODEL_ID,
    "stage": "base",
    "benchmark": "Uzbek Text Completion",

    "generation_config": {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": False,
        "repetition_penalty": 1.0
    },

    "total_prompts": len(PROMPTS),
    "results": base_completion_results
}

with open(
    RESULTS_DIR / "base_completion_results.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        baseline_completion,
        f,
        indent=4,
        ensure_ascii=False
    )

print("BASELINE TEXT COMPLETION SAVED SUCCESSFULLY")

In [ ]:
import json
from pathlib import Path

DATASET_PATH = Path("Cleaned 50k Raw/cleaned_002.jsonl")

assert DATASET_PATH.exists(), "Dataset not found!"

total_rows = 0
invalid_rows = 0
first_sample = None

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for line in f:
        try:
            row = json.loads(line)

            if first_sample is None:
                first_sample = row

            total_rows += 1

        except json.JSONDecodeError:
            invalid_rows += 1

print("=" * 60)
print("CPT DATASET INSPECTION")
print("=" * 60)

print(f"Dataset: {DATASET_PATH}")
print(f"Valid JSON rows: {total_rows:,}")
print(f"Invalid JSON rows: {invalid_rows:,}")

print("\nDATASET STRUCTURE:")

if first_sample:
    print(list(first_sample.keys()))

    print("\nSAMPLE:")
    print(
        json.dumps(
            first_sample,
            ensure_ascii=False,
            indent=2
        )[:2500]
    )

In [ ]:
SPLIT_DIR = Path("data_splits")

print("=" * 60)
print("EXISTING DATA SPLITS")
print("=" * 60)

if SPLIT_DIR.exists():
    for file in sorted(SPLIT_DIR.rglob("*")):
        if file.is_file():
            size_mb = file.stat().st_size / 1024**2

            print(
                f"{str(file):60s} "
                f"{size_mb:10.2f} MB"
            )
else:
    print("Data splits directory not found.")

In [ ]:
import json
import hashlib
from pathlib import Path

SPLIT_DIR = Path("data_splits")

split_paths = {
    "train": SPLIT_DIR / "train.jsonl",
    "validation": SPLIT_DIR / "validation.jsonl",
    "test": SPLIT_DIR / "test.jsonl",
}

split_hashes = {}
total_documents = 0

for split, path in split_paths.items():

    hashes = set()
    valid_rows = 0
    invalid_rows = 0
    duplicates = 0
    total_chars = 0

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                row = json.loads(line)
                text = row.get("text")

                if not isinstance(text, str) or not text.strip():
                    invalid_rows += 1
                    continue

                text = text.strip()

                text_hash = hashlib.sha256(
                    text.encode("utf-8")
                ).hexdigest()

                if text_hash in hashes:
                    duplicates += 1

                hashes.add(text_hash)

                valid_rows += 1
                total_chars += len(text)

            except json.JSONDecodeError:
                invalid_rows += 1

    split_hashes[split] = hashes
    total_documents += valid_rows

    print("=" * 60)
    print(f"SPLIT: {split.upper()}")
    print(f"Valid documents: {valid_rows:,}")
    print(f"Invalid documents: {invalid_rows:,}")
    print(f"Duplicate documents: {duplicates:,}")
    print(f"Total characters: {total_chars:,}")
    print(f"Average characters: {total_chars / max(valid_rows, 1):,.2f}")


print("\n" + "=" * 60)
print("DATA LEAKAGE CHECK")
print("=" * 60)

train_val = split_hashes["train"] & split_hashes["validation"]
train_test = split_hashes["train"] & split_hashes["test"]
val_test = split_hashes["validation"] & split_hashes["test"]

print(f"Train ↔ Validation: {len(train_val):,}")
print(f"Train ↔ Test: {len(train_test):,}")
print(f"Validation ↔ Test: {len(val_test):,}")

print(f"\nTotal documents: {total_documents:,}")

if not any([train_val, train_test, val_test]):
    print("\nEXACT DATA LEAKAGE CHECK: PASSED")
else:
    print("\nEXACT DATA LEAKAGE CHECK: FAILED")

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B"
MAX_SEQ_LENGTH = 1024
SEED = 42

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("=" * 60)
print("TOKENIZER INFORMATION")
print("=" * 60)

print(f"Model: {MODEL_NAME}")
print(f"Vocabulary size: {len(tokenizer):,}")
print(f"EOS token: {tokenizer.eos_token}")
print(f"EOS token ID: {tokenizer.eos_token_id}")
print(f"PAD token: {tokenizer.pad_token}")
print(f"PAD token ID: {tokenizer.pad_token_id}")
print(f"Sequence length: {MAX_SEQ_LENGTH}")

sample_text = "O‘zbekiston Respublikasi Markaziy Osiyoda joylashgan."

token_ids = tokenizer.encode(
    sample_text,
    add_special_tokens=False
)

print("\nSample text:", sample_text)
print("Token IDs:", token_ids)
print("Tokens:", tokenizer.convert_ids_to_tokens(token_ids))
print("Token count:", len(token_ids))
print("Decoded:", tokenizer.decode(token_ids))

In [ ]:
%pip install -U ipywidgets

In [ ]:
import json
import numpy as np

from pathlib import Path
from tqdm import tqdm

TRAIN_PATH = Path("data_splits/train.jsonl")

BATCH_SIZE = 128

token_lengths = []
batch_texts = []

def process_batch(texts):

    encoded = tokenizer(
        texts,
        add_special_tokens=False,
        truncation=False,
        padding=False,
        return_attention_mask=False,
        verbose=False
    )

    return [
        len(ids) + 1  # EOS token
        for ids in encoded["input_ids"]
    ]


with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    total_rows = sum(1 for _ in f)


with open(TRAIN_PATH, "r", encoding="utf-8") as f:

    for line in tqdm(
        f,
        total=total_rows,
        desc="Tokenizing documents"
    ):

        row = json.loads(line)
        text = row["text"].strip()

        if not text:
            continue

        batch_texts.append(text)

        if len(batch_texts) >= BATCH_SIZE:

            token_lengths.extend(
                process_batch(batch_texts)
            )

            batch_texts = []

    if batch_texts:
        token_lengths.extend(
            process_batch(batch_texts)
        )


lengths = np.array(token_lengths, dtype=np.int64)

print("\n" + "=" * 65)
print("TOKEN LENGTH STATISTICS")
print("=" * 65)

print(f"Total documents:      {len(lengths):,}")
print(f"Total tokens:         {lengths.sum():,}")
print(f"Average tokens:       {lengths.mean():,.2f}")
print(f"Median tokens:        {np.median(lengths):,.0f}")
print(f"Minimum tokens:       {lengths.min():,}")
print(f"Maximum tokens:       {lengths.max():,}")
print(f"Standard deviation:   {lengths.std():,.2f}")

print("\nPERCENTILES")
print("-" * 65)

for p in [25, 50, 75, 90, 95, 99]:
    print(f"P{p}: {np.percentile(lengths, p):,.0f} tokens")

print("\nSEQUENCE LENGTH ANALYSIS")
print("-" * 65)

for seq_len in [256, 512, 1024, 2048, 4096]:

    within = np.sum(lengths <= seq_len)
    above = np.sum(lengths > seq_len)

    print(f"\nSequence length: {seq_len}")

    print(
        f"Documents within: {within:,} "
        f"({within / len(lengths) * 100:.2f}%)"
    )

    print(
        f"Documents above:  {above:,} "
        f"({above / len(lengths) * 100:.2f}%)"
    )

    print(
        f"Ideal packed sequences: "
        f"{lengths.sum() // seq_len:,}"
    )

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))

plt.hist(
    lengths,
    bins=100,
    edgecolor="black",
    alpha=0.75
)

for seq_len in [512, 1024, 2048, 4096]:

    plt.axvline(
        seq_len,
        linestyle="--",
        label=f"Seq Length = {seq_len}"
    )

plt.xlim(0, np.percentile(lengths, 99))

plt.title("Qwen2.5-0.5B — Document Token Length Distribution")
plt.xlabel("Tokens per document")
plt.ylabel("Number of documents")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

In [ ]:
MAX_SEQ_LENGTH = 2048
TOKENIZE_BATCH_SIZE = 128

EOS_TOKEN_ID = tokenizer.eos_token_id
PAD_TOKEN_ID = tokenizer.pad_token_id

print("SEQUENCE CONFIGURATION")
print("=" * 60)

print(f"Sequence length: {MAX_SEQ_LENGTH}")
print(f"EOS token ID: {EOS_TOKEN_ID}")
print(f"PAD token ID: {PAD_TOKEN_ID}")
print(f"Tokenization batch size: {TOKENIZE_BATCH_SIZE}")

In [ ]:
import json
from pathlib import Path
from datasets import Dataset, DatasetDict
from tqdm import tqdm

SPLIT_DIR = Path("data_splits")

OUTPUT_DIR = Path(
    f"tokenized_cpt_{MAX_SEQ_LENGTH}"
)

split_paths = {
    "train": SPLIT_DIR / "train.jsonl",
    "validation": SPLIT_DIR / "validation.jsonl",
    "test": SPLIT_DIR / "test.jsonl"
}


def create_sequence(tokens):

    real_length = len(tokens)
    padding_length = MAX_SEQ_LENGTH - real_length

    input_ids = (
        tokens + [PAD_TOKEN_ID] * padding_length
    )

    attention_mask = (
        [1] * real_length + [0] * padding_length
    )

    labels = (
        tokens.copy() + [-100] * padding_length
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }


def generate_sequences(file_path):

    buffer = []
    text_batch = []

    def tokenize_documents(texts):

        result = tokenizer(
            texts,
            add_special_tokens=False,
            truncation=False,
            padding=False,
            return_attention_mask=False,
            verbose=False
        )

        return result["input_ids"]

    def process_documents(documents):

        nonlocal buffer

        for token_ids in documents:

            token_ids = token_ids + [EOS_TOKEN_ID]

            # Uzun documentlarni qismlarga ajratish
            for start in range(
                0,
                len(token_ids),
                MAX_SEQ_LENGTH
            ):

                chunk = token_ids[
                    start:start + MAX_SEQ_LENGTH
                ]

                # Document qismini bo‘sh joyga
                # to‘liq joylashtirishning iloji bo‘lmasa
                if len(buffer) + len(chunk) > MAX_SEQ_LENGTH:

                    if buffer:
                        yield create_sequence(buffer)

                    buffer = []

                buffer.extend(chunk)

                if len(buffer) == MAX_SEQ_LENGTH:

                    yield create_sequence(buffer)

                    buffer = []

    with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in tqdm(
            f,
            desc=f"Processing {Path(file_path).stem}"
        ):

            row = json.loads(line)
            text = row["text"].strip()

            if not text:
                continue

            text_batch.append(text)

            if len(text_batch) >= TOKENIZE_BATCH_SIZE:

                documents = tokenize_documents(text_batch)

                yield from process_documents(documents)

                text_batch = []

        if text_batch:

            documents = tokenize_documents(text_batch)

            yield from process_documents(documents)

    # Oxirgi to‘liq bo‘lmagan sequence
    if buffer:
        yield create_sequence(buffer)


tokenized_datasets = DatasetDict()

for split, path in split_paths.items():

    print(f"\nProcessing split: {split}")

    tokenized_datasets[split] = Dataset.from_generator(
        generate_sequences,
        gen_kwargs={"file_path": str(path)}
    )

tokenized_datasets.save_to_disk(str(OUTPUT_DIR))

print("\n" + "=" * 60)
print("TOKENIZED DATASET SAVED SUCCESSFULLY")
print("=" * 60)

print(tokenized_datasets)
print(f"\nSaved to: {OUTPUT_DIR.resolve()}")

In [ ]:
import numpy as np
from tqdm import tqdm

print("=" * 70)
print("PACKING QUALITY REPORT")
print("=" * 70)

for split, dataset in tokenized_datasets.items():

    total_sequences = len(dataset)
    total_slots = total_sequences * MAX_SEQ_LENGTH

    real_tokens = 0
    padding_tokens = 0
    eos_tokens = 0
    invalid_sequences = 0

    for batch in tqdm(
        dataset.iter(batch_size=256),
        desc=f"Checking {split}"
    ):

        ids = np.asarray(batch["input_ids"])
        masks = np.asarray(batch["attention_mask"])
        labels = np.asarray(batch["labels"])

        real_tokens += int(masks.sum())
        padding_tokens += int((masks == 0).sum())

        eos_tokens += int(
            ((ids == EOS_TOKEN_ID) & (masks == 1)).sum()
        )

        invalid = (
            ((masks == 0) & (labels != -100)) |
            ((masks == 1) & (labels != ids))
        )

        invalid_sequences += int(
            invalid.any(axis=1).sum()
        )

    efficiency = real_tokens / total_slots * 100

    print(f"\nSPLIT: {split.upper()}")
    print("-" * 70)

    print(f"Total sequences:    {total_sequences:,}")
    print(f"Total token slots:  {total_slots:,}")
    print(f"Real tokens:        {real_tokens:,}")
    print(f"Padding tokens:     {padding_tokens:,}")
    print(f"EOS tokens:         {eos_tokens:,}")
    print(f"Packing efficiency: {efficiency:.2f}%")
    print(f"Invalid sequences:  {invalid_sequences:,}")

    if split == "train":

        assert real_tokens == 91_792_959, \
            "Training token count mismatch!"

    assert invalid_sequences == 0, \
        "Invalid labels detected!"

print("\nPACKING VALIDATION PASSED")

In [ ]:
sample = tokenized_datasets["train"][0]

print("SEQUENCE INSPECTION")
print("=" * 60)

for key, value in sample.items():

    print(f"\n{key}")
    print(f"Length: {len(value)}")
    print(f"First 30 values: {value[:30]}")

print("\nDECODED TEXT")
print("=" * 60)

real_ids = [
    token_id
    for token_id, mask in zip(
        sample["input_ids"],
        sample["attention_mask"]
    )
    if mask == 1
]

print(
    tokenizer.decode(
        real_ids,
        skip_special_tokens=False
    )[:3000]
)

In [ ]:
import torch
import platform
import shutil
import psutil
import importlib.metadata as metadata

print("=" * 65)
print("TRAINING ENVIRONMENT")
print("=" * 65)

print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():

    props = torch.cuda.get_device_properties(0)
    free, total = torch.cuda.mem_get_info()

    print(f"\nGPU: {props.name}")
    print(f"Total GPU memory: {total / 1024**3:.2f} GiB")
    print(f"Free GPU memory: {free / 1024**3:.2f} GiB")
    print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")

ram = psutil.virtual_memory()

print("\nSYSTEM MEMORY")
print(f"Total RAM: {ram.total / 1024**3:.2f} GiB")
print(f"Available RAM: {ram.available / 1024**3:.2f} GiB")

disk = shutil.disk_usage(".")

print("\nDISK")
print(f"Total: {disk.total / 1024**3:.2f} GiB")
print(f"Free: {disk.free / 1024**3:.2f} GiB")

print("\nLIBRARY VERSIONS")

for library in [
    "unsloth",
    "unsloth_zoo",
    "transformers",
    "datasets",
    "accelerate",
    "trl",
    "peft",
    "bitsandbytes"
]:
    try:
        print(f"{library:20s}: {metadata.version(library)}")
    except metadata.PackageNotFoundError:
        print(f"{library:20s}: NOT INSTALLED")

In [ ]:
from unsloth import FastLanguageModel

import torch
import gc
from datasets import load_from_disk

gc.collect()
torch.cuda.empty_cache()

MODEL_NAME = "Qwen/Qwen2.5-0.5B"
MAX_SEQ_LENGTH = 2048
SEED = 42

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.bfloat16,
    load_in_4bit=False,
    load_in_8bit=False,
    full_finetuning=False,
)

tokenized_datasets = load_from_disk(
    "tokenized_cpt_2048"
)

print("=" * 65)
print("MODEL LOADED SUCCESSFULLY")
print("=" * 65)

print(f"Model: {MODEL_NAME}")
print(f"Model dtype: {next(model.parameters()).dtype}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

print(
    f"GPU allocated: "
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GiB"
)

print(
    f"GPU reserved: "
    f"{torch.cuda.memory_reserved() / 1024**3:.2f} GiB"
)

print(tokenized_datasets)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],

    lora_alpha=32,
    lora_dropout=0,
    bias="none",

    use_gradient_checkpointing="unsloth",

    random_state=SEED,

    use_rslora=False,
    loftq_config=None,
)

model.config.use_cache = False

print("=" * 65)
print("LORA CONFIGURATION")
print("=" * 65)

model.print_trainable_parameters()

print(
    f"\nGPU allocated: "
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GiB"
)

print(
    f"GPU reserved: "
    f"{torch.cuda.memory_reserved() / 1024**3:.2f} GiB"
)

In [ ]:
import math

model.train()
model.zero_grad(set_to_none=True)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

sample = tokenized_datasets["train"][0]

batch = {
    key: torch.tensor(
        [sample[key]],
        dtype=torch.long,
        device="cuda"
    )
    for key in [
        "input_ids",
        "attention_mask",
        "labels"
    ]
}

print("Running forward pass...")

with torch.autocast(
    device_type="cuda",
    dtype=torch.bfloat16
):
    outputs = model(**batch)
    loss = outputs.loss

loss_value = loss.item()

assert math.isfinite(loss_value), "Invalid loss!"

print(f"Forward loss: {loss_value:.4f}")

print("Running backward pass...")

loss.backward()

torch.cuda.synchronize()

print("\n" + "=" * 65)
print("SMOKE TEST RESULTS")
print("=" * 65)

print(f"Loss: {loss_value:.4f}")

print(
    f"Peak GPU allocated: "
    f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB"
)

print(
    f"Peak GPU reserved: "
    f"{torch.cuda.max_memory_reserved() / 1024**3:.2f} GiB"
)

free, total = torch.cuda.mem_get_info()

print(f"Free VRAM: {free / 1024**3:.2f} GiB")
print(f"Total VRAM: {total / 1024**3:.2f} GiB")

model.zero_grad(set_to_none=True)

del outputs, loss, batch

gc.collect()
torch.cuda.empty_cache()

print("\nFORWARD & BACKWARD TEST PASSED")

In [ ]:
model.print_trainable_parameters()

In [ ]:
total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

frozen_params = total_params - trainable_params

print("=" * 60)
print("MODEL PARAMETER STATISTICS")
print("=" * 60)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {frozen_params:,}")
print(f"Trainable percentage: {100 * trainable_params / total_params:.4f}%")
print(f"Frozen percentage:    {100 * frozen_params / total_params:.4f}%")

In [ ]:
from transformers import (
    Trainer,
    TrainingArguments,
    default_data_collator
)

training_args = TrainingArguments(

    output_dir="./cpt_smoke_test",

    # Batch configuration
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,

    # Training configuration
    max_steps=20,
    learning_rate=5e-5,
    warmup_steps=2,
    lr_scheduler_type="cosine",

    # Optimizer
    optim="adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=1.0,

    # Precision
    bf16=True,
    fp16=False,

    # Memory optimization
    gradient_checkpointing=True,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,

    # Logging
    logging_steps=1,
    logging_first_step=True,
    report_to="none",

    # Evaluation
    eval_strategy="no",

    # Checkpoint
    save_strategy="no",

    # Reproducibility
    seed=42,
    data_seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    data_collator=default_data_collator,
)

print("CPT TRAINER READY")
model.print_trainable_parameters()

In [ ]:
import time
import gc
import torch

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

start_time = time.time()

train_result = trainer.train()

torch.cuda.synchronize()

elapsed = time.time() - start_time

print("\n" + "=" * 65)
print("CPT TRAINING SMOKE TEST RESULTS")
print("=" * 65)

print(f"Training loss: {train_result.training_loss:.4f}")
print(f"Optimizer steps: {trainer.state.global_step}")
print(f"Elapsed time: {elapsed / 60:.2f} minutes")

print(
    f"Peak GPU allocated: "
    f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB"
)

print(
    f"Peak GPU reserved: "
    f"{torch.cuda.max_memory_reserved() / 1024**3:.2f} GiB"
)

free, total = torch.cuda.mem_get_info()

print(f"Free VRAM: {free / 1024**3:.2f} GiB")
print(f"Total VRAM: {total / 1024**3:.2f} GiB")

print("\nTRAINING METRICS:")

for key, value in train_result.metrics.items():
    print(f"{key}: {value}")

In [ ]:
import json
import hashlib
import numpy as np
from pathlib import Path

hashes = {}

for split in ["train", "validation", "test"]:

    path = Path(f"data_splits/{split}.jsonl")

    with open(path, encoding="utf-8") as f:

        hashes[split] = {
            hashlib.sha256(
                json.loads(line)["text"].strip().encode("utf-8")
            ).hexdigest()
            for line in f
        }

assert not hashes["train"] & hashes["validation"]
assert not hashes["train"] & hashes["test"]
assert not hashes["validation"] & hashes["test"]

print("EXACT DATA LEAKAGE CHECK: PASSED")

for split, dataset in tokenized_datasets.items():

    real_tokens = 0
    padding_tokens = 0

    for batch in dataset.iter(batch_size=128):

        ids = np.asarray(batch["input_ids"])
        mask = np.asarray(batch["attention_mask"])
        labels = np.asarray(batch["labels"])

        assert ids.shape[1] == 2048
        assert ids.shape == mask.shape == labels.shape
        assert np.isin(mask, [0, 1]).all()
        assert (labels[mask == 0] == -100).all()
        assert (labels[mask == 1] == ids[mask == 1]).all()

        real_tokens += int(mask.sum())
        padding_tokens += int((mask == 0).sum())

    efficiency = (
        real_tokens / (real_tokens + padding_tokens) * 100
    )

    print(f"\nSPLIT: {split.upper()}")
    print(f"Sequences: {len(dataset):,}")
    print(f"Real tokens: {real_tokens:,}")
    print(f"Padding tokens: {padding_tokens:,}")
    print(f"Packing efficiency: {efficiency:.2f}%")

    if split == "train":
        assert real_tokens == 91_792_959

print("\nDATASET VALIDATION PASSED")

In [ ]:
from transformers import (
    Trainer,
    TrainingArguments,
    default_data_collator
)

OUTPUT_DIR = "./Qwen2.5-0.5B-Uzbek-CPT-LoRA"

training_args = TrainingArguments(

    output_dir=OUTPUT_DIR,

    # Batch configuration
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,

    # Training
    num_train_epochs=1,
    learning_rate=5e-5,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",

    # Optimizer
    optim="adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=1.0,

    # Precision
    bf16=True,
    fp16=False,

    # Memory optimization
    gradient_checkpointing=True,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,

    # Logging
    logging_steps=25,
    logging_first_step=True,
    report_to="none",

    # Validation
    eval_strategy="steps",
    eval_steps=250,
    prediction_loss_only=True,

    # Checkpoint
    save_strategy="steps",
    save_steps=250,
    save_total_limit=3,
    save_only_model=False,

    # Best model
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Reproducibility
    seed=42,
    data_seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=default_data_collator,
)

print("FULL DATASET CPT TRAINER READY")

model.print_trainable_parameters()

In [ ]:
import time
import gc
import torch

from transformers.trainer_utils import get_last_checkpoint

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

checkpoint = get_last_checkpoint(OUTPUT_DIR)

if checkpoint:
    print(f"Resuming from: {checkpoint}")
else:
    print("Starting new CPT training")

start_time = time.time()

train_result = trainer.train(
    resume_from_checkpoint=checkpoint
)

torch.cuda.synchronize()

elapsed = time.time() - start_time

print("\n" + "=" * 65)
print("CPT TRAINING COMPLETED")
print("=" * 65)

print(f"Training loss: {train_result.training_loss:.4f}")
print(f"Total steps: {trainer.state.global_step:,}")
print(f"Elapsed time: {elapsed / 3600:.2f} hours")

print(
    f"Peak GPU allocated: "
    f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB"
)

trainer.save_metrics("train", train_result.metrics)
trainer.save_state()

In [ ]:
FINAL_ADAPTER_DIR = "./Qwen2.5-0.5B-Uzbek-CPT-Final"

model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)

print("CPT LORA ADAPTER SAVED SUCCESSFULLY")
print(f"Saved to: {FINAL_ADAPTER_DIR}")